### Importación, configuración y procesamiento del dataset

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset, DatasetDict
from transformers import BertModel, AutoTokenizer
import matplotlib.pyplot as plt
import numpy as np

# 1. Configurar el dispositivo de cómputo (GPU si está disponible, de lo contrario CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# 2. Cargar el dataset utilizando el namespace correcto
dataset = load_dataset('stanfordnlp/imdb')

# 3. Reducir el tamaño usando el rango de 10,000 a 15,000 para incluir ambas categorías
small_data = DatasetDict({
    split: dataset[split].select(range(10000, 15000)) for split in ['train', 'test']
})

# 4. Cargar el tokenizador preentrenado de BERT
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# 5. Definir la función de tokenización con longitud fija de 512 tokens
def tokenize_function(one_sample):
    return tokenizer(
        one_sample['text'],
        max_length=512,          # Longitud máxima de secuencia para BERT
        padding='max_length',    # Rellenar con ceros (tokens de padding) si es más corto
        truncation=True          # Truncar si excede los 512 tokens
    )

# 6. Aplicar la función de tokenización al dataset reducido por lotes para mayor eficiencia
tokenized_dataset = small_data.map(tokenize_function, batched=True)

# 7. Eliminar la columna de texto original y establecer el formato de tensores para PyTorch
tokenized_dataset = tokenized_dataset.remove_columns(['text'])
tokenized_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# 8. Crear los DataLoaders para entrenamiento (mezclando los datos) y prueba
train_dataloader = DataLoader(tokenized_dataset['train'], shuffle=True, batch_size=8)
test_dataloader = DataLoader(tokenized_dataset['test'], batch_size=32)

print("¡Parte 1 configurada correctamente con el dataset y rango exactos!")

Usando dispositivo: cuda


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

¡Parte 1 configurada correctamente con el dataset y rango exactos!


### Creación del modelo personalizado y congelación de pesos

In [3]:
# 1. Definir la clase personalizada para la clasificación binaria con BERT
class BertForBinaryClassification(nn.Module):
    def __init__(self, num_labels=2):
        super(BertForBinaryClassification, self).__init__()
        # Cargar el modelo base BERT preentrenado (Encoder-only)
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        # Capa de dropout y capa lineal para la clasificación
        self.dropout = nn.Dropout(self.bert.config.hidden_dropout_prob)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        # Pasada hacia adelante por el modelo BERT base
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # Tomar la salida agrupada (pooled output correspondiente al token [CLS])
        pooled_output = outputs[1]

        # Aplicar dropout y la capa clasificadora para obtener los logits
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        return logits

# 2. Instanciar el modelo y moverlo al dispositivo configurado (GPU/CPU)
model = BertForBinaryClassification().to(device)

# 3. Congelar los pesos del modelo base BERT para entrenar únicamente la cabeza de clasificación
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

print("¡Parte 2 completada: Modelo creado y pesos congelados correctamente!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


¡Parte 2 completada: Modelo creado y pesos congelados correctamente!


### Configuración del entrenamiento y bucle de optimización

In [4]:
# 1. Configurar el número de épocas, el optimizador y la función de pérdida
num_epochs = 1
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
loss_fun = nn.CrossEntropyLoss()

print("¡Iniciando el ciclo de entrenamiento y evaluación!\n")

# 2. Bucle principal de entrenamiento (Fine-tuning)
for epoch in range(num_epochs):
    model.train()  # Poner el modelo en modo de entrenamiento
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, batch in enumerate(train_dataloader):
        # Mover los tensores del lote al dispositivo (GPU/CPU)
        tokenz = batch['input_ids'].to(device)
        att_msk = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # Limpiar los gradientes anteriores
        optimizer.zero_grad()

        # Pasada hacia adelante (Forward pass) para obtener los logits
        logits = model(tokenz, attention_mask=att_msk)

        # Calcular la pérdida del lote
        loss = loss_fun(logits, labels)

        # Pasada hacia atrás (Backward pass) y optimización de los pesos
        loss.backward()
        optimizer.step()

        # Acumular estadísticas de entrenamiento
        train_loss += loss.item()
        predLabels = torch.argmax(logits, dim=1)
        train_correct += (predLabels == labels).sum().item()
        train_total += labels.size(0)

        # Mostrar progreso cada 100 lotes
        if (batch_idx + 1) % 100 == 0:
            print(f"Época [{epoch+1}/{num_epochs}], Lote [{batch_idx+1}/{len(train_dataloader)}] - Pérdida actual: {loss.item():.4f}")

    # Calcular métricas globales de la época de entrenamiento
    epoch_train_loss = train_loss / len(train_dataloader)
    epoch_train_acc = train_correct / train_total
    print(f"\n--- Fin de la Época {epoch+1} ---")
    print(f"Entrenamiento -> Pérdida: {epoch_train_loss:.4f} | Precisión: {100 * epoch_train_acc:.2f}%")

    # 3. Fase de Evaluación con el conjunto de prueba (Test)
    model.eval()  # Poner el modelo en modo de evaluación
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():  # Desactivar el cálculo de gradientes para ahorrar memoria
        for batch in test_dataloader:
            tokenz = batch['input_ids'].to(device)
            att_msk = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(tokenz, attention_mask=att_msk)
            loss = loss_fun(logits, labels)

            test_loss += loss.item()
            predLabels = torch.argmax(logits, dim=1)
            test_correct += (predLabels == labels).sum().item()
            test_total += labels.size(0)

    # Calcular métricas globales de evaluación
    epoch_test_loss = test_loss / len(test_dataloader)
    epoch_test_acc = test_correct / test_total
    print(f"Prueba (Test)    -> Pérdida: {epoch_test_loss:.4f} | Precisión: {100 * epoch_test_acc:.2f}%\n")

print("Proceso de fine-tuning y evaluación finalizado con éxito!")

¡Iniciando el ciclo de entrenamiento y evaluación!

Época [1/1], Lote [100/625] - Pérdida actual: 0.6851
Época [1/1], Lote [200/625] - Pérdida actual: 0.7003
Época [1/1], Lote [300/625] - Pérdida actual: 0.6549
Época [1/1], Lote [400/625] - Pérdida actual: 0.6957
Época [1/1], Lote [500/625] - Pérdida actual: 0.6597
Época [1/1], Lote [600/625] - Pérdida actual: 0.6872

--- Fin de la Época 1 ---
Entrenamiento -> Pérdida: 0.7029 | Precisión: 47.58%
Prueba (Test)    -> Pérdida: 0.6957 | Precisión: 50.06%

Proceso de fine-tuning y evaluación finalizado con éxito!
